In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

if 'df' not in locals() and 'df' not in globals():
    if 'data' in locals() or 'data' in globals():
        df = data  
    else:
        print("Змінну df не знайдено. Спроба автоматичного завантаження...")
        try:
            df = load_dataset() 
        except NameError:
            df = pd.read_csv("application_train.csv")  

# Створення вихідної папки для збереження графіків
output_folder = "all_plots_presentation"
os.makedirs(output_folder, exist_ok=True)  # якщо папка вже існує — не перезаписуємо

# Розрахунок похідних аналітичних ознак
df_all = df.copy()  # працюємо з копією, щоб не змінювати оригінальний датафрейм

# Вік у роках, DAYS_BIRTH зберігається як від'ємне число днів
if 'DAYS_BIRTH' in df_all.columns and 'AGE' not in df_all.columns:
    df_all['AGE'] = -df_all['DAYS_BIRTH'] / 365.25

# Стаж роботи у роках, значення 365243 — маркерне значення для безробітних, замінюємо його на NaN 
if 'DAYS_EMPLOYED' in df_all.columns and 'EMPLOYMENT_YEARS' not in df_all.columns:
    df_all['EMPLOYMENT_YEARS'] = df_all['DAYS_EMPLOYED'].replace(365243, np.nan) / -365.25

# Відношення річного платежу до доходу — показник боргового навантаження
if 'AMT_ANNUITY' in df_all.columns and 'AMT_INCOME_TOTAL' in df_all.columns and 'ANNUITY_TO_INCOME' not in df_all.columns:
    df_all['ANNUITY_TO_INCOME'] = df_all['AMT_ANNUITY'] / df_all['AMT_INCOME_TOTAL']
    df_all['ANNUITY_TO_INCOME'] = df_all['ANNUITY_TO_INCOME'].replace([np.inf, -np.inf], np.nan)  # усуваємо ділення на нуль

# Відбір числових ознак для побудови графіків
excluded_cols = ['SK_ID_CURR', 'TARGET']  # технічний ID та цільова змінна не потрібні
numeric_cols = [col for col in df_all.select_dtypes(include=[np.number]).columns if col not in excluded_cols]

print(f"Знайдено числових ознак для аналізу: {len(numeric_cols)}")
print(f"Усі графіки будуть збережені в папку: '{output_folder}/'\n")

# Генерація boxplot для кожної числової ознаки
generated_count = 0

for col in numeric_cols:
    temp_df = df_all[[col, 'TARGET']].dropna()  # видаляє рядки з NaN 
    if temp_df.empty:
        continue  

    
    p95 = temp_df[col].quantile(0.95)
    p99 = temp_df[col].quantile(0.99)
    max_val = temp_df[col].max()

    if max_val > p95 * 5 and p99 > 0:
        temp_df = temp_df[temp_df[col] <= p99]  

    plt.figure(figsize=(4, 5.5))  

    try:
        ax = sns.boxplot(
            data=temp_df,
            x='TARGET',
            y=col,
            color='royalblue',       
            width=0.4,                
            linewidth=1.2,
            boxprops=dict(edgecolor='black'),
            whiskerprops=dict(color='black'),
            capprops=dict(color='black'),
            medianprops=dict(color='orange', linewidth=2),  # медіана виділена контрастним кольором
            showfliers=True,          # показуємо викиди за вусами
            flierprops=dict(
                marker='o',
                markerfacecolor='dimgray',
                markersize=3.5,
                markeredgecolor='none',
                alpha=0.4             
            )
        )

        clean_title = col.replace('_', ' ')  
        ax.set_title(clean_title, pad=15, fontweight='bold', fontsize=13)
        ax.set_xlabel('TARGET', labelpad=10, fontsize=11)
        ax.set_ylabel('Значення показника', labelpad=10, fontsize=11)
        ax.set_xticklabels(['TARGET = 0', 'TARGET = 1'], fontsize=11)  

        ax.grid(axis='y', linestyle='--', alpha=0.5) 
        ax.set_axisbelow(True) 
        plt.tight_layout()
        filename = os.path.join(output_folder, f"plot_{col.lower()}.png")
        plt.savefig(filename, dpi=300, bbox_inches='tight')  
        plt.close()

        generated_count += 1
        if generated_count % 10 == 0:
            print(f"Успішно згенеровано {generated_count} графіків...") 

    except Exception as e:
        plt.close()  
        continue     

print(f"\nГотово! Всього побудовано та збережено {generated_count} графіків у папку '{output_folder}'.")

Знайдено числових ознак для аналізу: 107
Усі графіки будуть збережені в папку: 'all_plots_presentation/'



C:\Users\snp22\AppData\Local\Temp\ipykernel_18060\1212114476.py:93: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(['TARGET = 0', 'TARGET = 1'], fontsize=11)
C:\Users\snp22\AppData\Local\Temp\ipykernel_18060\1212114476.py:93: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(['TARGET = 0', 'TARGET = 1'], fontsize=11)
C:\Users\snp22\AppData\Local\Temp\ipykernel_18060\1212114476.py:93: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(['TARGET = 0', 'TARGET = 1'], fontsize=11)
C:\Users\snp22\AppData\Local\Temp\ipykernel_18060\1212114476.py:93: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(['T

Успішно згенеровано 10 графіків...


C:\Users\snp22\AppData\Local\Temp\ipykernel_18060\1212114476.py:93: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(['TARGET = 0', 'TARGET = 1'], fontsize=11)
C:\Users\snp22\AppData\Local\Temp\ipykernel_18060\1212114476.py:93: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(['TARGET = 0', 'TARGET = 1'], fontsize=11)
C:\Users\snp22\AppData\Local\Temp\ipykernel_18060\1212114476.py:93: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(['TARGET = 0', 'TARGET = 1'], fontsize=11)
C:\Users\snp22\AppData\Local\Temp\ipykernel_18060\1212114476.py:93: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(['T

Успішно згенеровано 20 графіків...


...

C:\Users\snp22\AppData\Local\Temp\ipykernel_18060\1212114476.py:93: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(['TARGET = 0', 'TARGET = 1'], fontsize=11)
C:\Users\snp22\AppData\Local\Temp\ipykernel_18060\1212114476.py:93: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(['TARGET = 0', 'TARGET = 1'], fontsize=11)
C:\Users\snp22\AppData\Local\Temp\ipykernel_18060\1212114476.py:93: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(['TARGET = 0', 'TARGET = 1'], fontsize=11)
C:\Users\snp22\AppData\Local\Temp\ipykernel_18060\1212114476.py:93: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(['T

Успішно згенеровано 100 графіків...


C:\Users\snp22\AppData\Local\Temp\ipykernel_18060\1212114476.py:93: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(['TARGET = 0', 'TARGET = 1'], fontsize=11)
C:\Users\snp22\AppData\Local\Temp\ipykernel_18060\1212114476.py:93: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(['TARGET = 0', 'TARGET = 1'], fontsize=11)
C:\Users\snp22\AppData\Local\Temp\ipykernel_18060\1212114476.py:93: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(['TARGET = 0', 'TARGET = 1'], fontsize=11)
C:\Users\snp22\AppData\Local\Temp\ipykernel_18060\1212114476.py:93: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(['T


Готово! Всього побудовано та збережено 107 графіків у папку 'all_plots_presentation'.


C:\Users\snp22\AppData\Local\Temp\ipykernel_18060\1212114476.py:93: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(['TARGET = 0', 'TARGET = 1'], fontsize=11)
